# GFP 결합체 설계와 형광 변화 평가 · v0.1.6

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/GFP_Binder_T4.ipynb?v=0.1.6)

GFP 결합체의 뼈대와 서열을 설계하고 복합체 구조·형광 변화 가능성을 평가합니다. 대조군 실험은 새 후보를 평가할 조건의 신뢰도를 확인하는 단계입니다. T4 GPU 런타임에서 아래 실행 모드를 선택한 뒤 전체 셀을 실행합니다.

| RUN_MODE | 실행 범위 |
| --- | --- |
| full | RFD3 뼈대 생성 → LigandMPNN 서열 설계 → RF3 복합체 예측 → 형광 평가 |
| benchmark | 알려진 두 대조군의 결합 자세 재현 |
| followup | Minimizer 시드 확대와 enhancer 결합체 템플릿 제거 비교 |

현재 기본값은 `followup`입니다. 두 실험을 모두 선택하면 13개 예측이며, 실행 04의 소요시간 기준 추론에 약 40분입니다. 설치·다운로드 시간은 별도입니다.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="gfp_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")

## 실행 설정

- `followup`: `EXPERIMENT`에서 `all`, `minimizer_seed_search`, `enhancer_target_only`를 선택합니다. Minimizer는 `both`·시드 0–9, enhancer는 `target`·시드 0·1·2이며 모두 recycle 10회입니다.
- `benchmark`: `BENCHMARK_TEMPLATE`, `BENCHMARK_RECYCLES`, `BENCHMARK_SEEDS`를 사용합니다. `target`은 GFP만, `both`는 두 단백질의 개별 구조를 제공합니다. 체인 간 정답 거리는 가립니다.
- `full`: 자체 형광 측정 CSV가 있으면 `ASSAY_CSV`에 경로를 입력합니다. CSV에는 `id,ratio,group,assay,source`와 `sequence` 또는 `pdb,entity`가 필요합니다. `ratio`는 동일 실험에서 측정한 기준 대비 형광 배율이며 `group`은 유사 결합체의 그룹입니다. 경로를 비워 두면 공개 대조군을 사용합니다.


In [ ]:
import json

RUN_MODE = "followup"  # @param ["full", "benchmark", "followup"]
EXPERIMENT = "all"  # @param ["all", "minimizer_seed_search", "enhancer_target_only"]
BENCHMARK_TEMPLATE = "both"  # @param ["target", "both"]
BENCHMARK_RECYCLES = 10  # @param [3, 10] {type:"raw"}
BENCHMARK_SEEDS = "0,1,2"  # @param {type:"string"}
ASSAY_CSV = ""  # @param {type:"string"}

jobs = []
if RUN_MODE == "followup":
    settings = json.loads((checkout / "gfp_pipeline/benchmark_experiments.json").read_text(encoding="utf-8"))
    experiments = [item for item in settings["experiments"]
                   if EXPERIMENT == "all" or item["id"] == EXPERIMENT]
    if not experiments:
        raise ValueError("실행할 실험을 선택하세요.")
    for experiment in experiments:
        jobs.append({"experiment": experiment, "args": [
            "--benchmark-only", "--benchmark-controls", experiment["control"],
            "--benchmark-template", experiment["template"],
            "--benchmark-recycles", str(experiment["recycles"]),
            "--benchmark-seeds", *map(str, experiment["seeds"])]})
        print(experiment["label"], "recycles:", experiment["recycles"], "seeds:", experiment["seeds"])
    print("총 예측 수:", sum(len(item["seeds"]) for item in experiments))
elif RUN_MODE == "benchmark":
    seeds = [int(value.strip()) for value in BENCHMARK_SEEDS.split(",")]
    jobs.append({"experiment": {"id": "benchmark", "label": "기본 대조군 평가"}, "args": [
        "--benchmark-only", "--benchmark-template", BENCHMARK_TEMPLATE,
        "--benchmark-recycles", str(BENCHMARK_RECYCLES), "--benchmark-seeds", *map(str, seeds)]})
elif RUN_MODE == "full":
    args = ["--assays", str(Path(ASSAY_CSV).expanduser().resolve(strict=True))] if ASSAY_CSV.strip() else []
    jobs.append({"experiment": {"id": "full", "label": "전체 설계·형광 평가"}, "args": args})
else:
    raise ValueError("실행 모드를 선택하세요.")


In [ ]:
completed_runs = []
for job in jobs:
    experiment = job["experiment"]
    print(f"실행: {experiment['label']}", flush=True)
    command = [sys.executable, str(checkout / "gfp_pipeline/run.py"),
               "--revision", revision, *job["args"]]
    result_zip = None
    with subprocess.Popen(command, stdout=subprocess.PIPE, text=True) as process:
        for line in process.stdout:
            print(line.rstrip(), flush=True)
            if line.rstrip().endswith(".zip"):
                result_zip = Path(line.strip())
        if process.wait():
            raise RuntimeError(f"{experiment['id']}: 실행 실패. 위 로그를 확인하세요.")
    if result_zip is None or not result_zip.is_file():
        raise RuntimeError(f"{experiment['id']}: 결과 ZIP을 찾을 수 없습니다.")
    completed_runs.append({"experiment": experiment, "result_zip": result_zip,
                           "result_dir": result_zip.with_suffix("")})
    print(f"완료: {experiment['label']} — {result_zip.name}", flush=True)


## 결과 확인

대조군은 기존 접촉면 기준과 GFP 정렬 후 결합체 RMSD, Cα 잔기 쌍 접촉 회복률, ipTM 순위를 함께 확인합니다. RMSD는 낮을수록, 회복률은 높을수록 기준 자세와 가깝습니다. 후속 비교는 실행 04와 나란히 표시합니다. 단독 대조군 실행은 이전 두 대조군 연속 실행과 난수 소비 순서가 달라질 수 있습니다.

전체 설계 모드는 구조·실험 데이터 조건을 통과한 후보에 형광 배율을 표시합니다. 배율 1.0은 기준과 같은 형광이며, 보류된 경우 사유가 함께 표시됩니다. 마지막 셀에서 결과 ZIP을 내려받습니다.


In [ ]:
from IPython.display import display

if RUN_MODE == "followup":
    import pandas as pd
    from IPython.display import display

    baseline = json.loads((checkout / settings["baseline_summary"]).read_text(encoding="utf-8"))
    comparison = []
    for run in completed_runs:
        experiment = run["experiment"]
        current = json.loads((run["result_dir"] / "metrics/benchmark_summary.json").read_text(encoding="utf-8"))
        for label, report in (("실행 04", baseline), (experiment["id"], current)):
            for row in report["controls"]:
                if row["control"] == experiment["control"]:
                    comparison.append({"run": label, "template": report["template_mode"], **row})
        poses = pd.read_csv(run["result_dir"] / "metrics/benchmark_poses.csv")
        print(experiment["label"])
        display(poses[["id", "seed", "iptm_rank", "iptm", "reference_pose_recovered",
                       "benchmark_contact_jaccard", "binder_ca_rmsd_after_target_alignment_angstrom",
                       "native_ca_contact_pair_recovery", "has_clash"]].round(3))
    comparison_table = pd.DataFrame(comparison)
    display(comparison_table[["run", "control", "template", "poses_tested", "poses_recovered",
                              "top_iptm", "top_iptm_binder_rmsd_angstrom",
                              "top_iptm_native_ca_contact_pair_recovery",
                              "best_binder_rmsd_angstrom", "best_recovered_iptm_rank"]].round(3))

else:
    result_dir = completed_runs[0]["result_dir"]
    import json
    import pandas as pd

    if (result_dir / "metrics/benchmark_summary.csv").is_file():
        config = json.loads((result_dir / "config/source.json").read_text(encoding="utf-8"))
        print(f"템플릿: {config['benchmark_template']}, recycles: {config['rf3']['n_recycles']}, seeds: {config['rf3_seeds']}")
        summary = pd.read_csv(result_dir / "metrics/benchmark_summary.csv")
        summary["status"] = summary["status"].map({
            "no_pose_recovered": "기준 충족 자세 없음",
            "pose_ranked_lower": "기준 충족 자세가 하위에 있음",
            "top_pose_recovered": "1위 자세가 기준 충족",
        })
        display(summary[["control", "status", "poses_recovered", "top_iptm_seed",
                         "top_iptm_contact_jaccard", "best_contact_jaccard",
                         "best_recovered_iptm_rank", "top_iptm_binder_rmsd_angstrom",
                         "top_iptm_native_ca_contact_pair_recovery"]].round(3))
        poses = pd.read_csv(result_dir / "metrics/benchmark_poses.csv")
        display(poses[["id", "seed", "iptm_rank", "iptm", "benchmark_contact_jaccard",
                       "target_ca_rmsd_angstrom",
                       "binder_ca_rmsd_after_target_alignment_angstrom",
                       "native_ca_contact_pair_recovery", "has_clash"]].round(3))
    else:
        metrics = json.loads((result_dir / "metrics/fluorescence_model.json").read_text(encoding="utf-8"))
        status_text = {
            "ready": "형광 모델 검증 통과",
            "structural_benchmark_failed": "대조군 구조 검증 미통과",
            "insufficient_control_groups": "독립 실험 그룹 부족",
            "baseline_not_improved": "형광 변화 없음 기준 대비 개선 없음",
            "candidate_structure_failed": "후보 구조 검증 미통과",
            "outside_training_range": "접촉 특징이 학습 범위 밖",
            "outside_observed_effect_range": "예측값이 실험에서 관측한 범위 밖",
            "predicted": "예측값 산출",
        }
        print(status_text[metrics['validation']])
        if "held_out_mae_log2" in metrics:
            print(f"그룹별 검증 MAE (log2): {metrics['held_out_mae_log2']:.3f}")
            print(f"변화 없음 기준 MAE (log2): {metrics['no_effect_mae_log2']:.3f}")

        results = pd.read_csv(result_dir / "metrics/fluorescence_predictions.csv")
        results["prediction_status"] = results["prediction_status"].map(status_text)
        ratio = results["predicted_fluorescence_ratio"].round(3)
        results["predicted_fluorescence_ratio"] = ratio.astype(object).where(ratio.notna(), "보류")
        display(results[["id", "iptm", "prediction_status", "predicted_fluorescence_ratio"]])

In [ ]:
from google.colab import files

if RUN_MODE == "followup":
    from zipfile import ZipFile, ZIP_DEFLATED
    from google.colab import files

    bundle_dir = Path(mkdtemp(prefix="gfp_followup_", dir="/content"))
    bundle = bundle_dir.with_suffix(".zip")
    manifest = {"revision": revision, "baseline_summary": settings["baseline_summary"],
                "runs": [{"experiment": run["experiment"], "original_zip": run["result_zip"].name}
                         for run in completed_runs]}
    with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as archive:
        archive.writestr("experiments.json", json.dumps(manifest, ensure_ascii=False, indent=2))
        for run in completed_runs:
            for path in sorted(run["result_dir"].rglob("*")):
                if path.is_file():
                    name = Path(run["experiment"]["id"]) / path.relative_to(run["result_dir"])
                    archive.write(path, arcname=name.as_posix())
    print(bundle)
    files.download(str(bundle))

else:
    files.download(str(completed_runs[0]["result_zip"]))
